# SESSION 5: Data Collection using Pandas (Part 2)
## Assignment Track 3: Data Science Foundations
**Case Study:** Annual College Cultural & Tech Festival (*Vivanta Fest 2025*) Operations Engine  
**Author:** Atharv Chaudhari (`ds-work`)

---

### Topics Covered:
1. **Reading data from relational databases:** SQLite / MySQL / PostgreSQL via `pd.read_sql()` and `pd.read_sql_query()`
2. **Semi-structured data flattening:** Parsing nested JSON API responses using `pd.json_normalize()`
3. **Web-based dataset extraction:** Ingesting live CSV datasets directly from raw GitHub URLs
4. **Modern file system automation with `pathlib`:** Directory creation, batch pattern globbing, and dynamic discovery
5. **Multi-table integration:** Horizontal concatenation (`pd.concat`) and relational joins (`pd.merge`) with cardinality validation (`validate='m:1'`)


---
## Task 1: SQLite Database Setup & SQL Query Ingestion

**Objectives:**
- Create a SQLite database `fest.db` with `sqlite3.connect('fest.db')`.
- Generate a 400-row `registrations` table using `np.random.seed(42)`:
  - `reg_id`: Registration IDs `R001` through `R400`.
  - `student_id`: Student IDs `S001` through `S400`.
  - `event`: Event categories from `['Dance', 'Coding', 'Quiz', 'Gaming', 'Music']`.
  - `college`: Participant institutions from 6 invented college names.
  - `ticket_inr`: Ticket tiers from 0 to 500 in steps of 50 (`range(0, 501, 50)`).
  - `paid`: Payment settlement status (`'Yes'` or `'No'`).
- Persist the dataset to `fest.db` using `to_sql('registrations', conn, index=False)`.
- Query paid rows back using two distinct Pandas SQL methods:
  1. `pd.read_sql_query("SELECT * FROM registrations WHERE paid = 'Yes'", conn)`
  2. `pd.read_sql("SELECT * FROM registrations WHERE paid = 'Yes'", conn)`
- Print the shapes of both resulting DataFrames.
- *Constraint applied:* **Use only a simple `SELECT ... WHERE` query.**


In [1]:
import sqlite3
import numpy as np
import pandas as pd

# Set seed for exact reproducibility
np.random.seed(42)

# Generate 400 registration entries
student_ids = [f"S{i:03d}" for i in range(1, 401)]
reg_ids = [f"R{i:03d}" for i in range(1, 401)]
events_pool = ['Dance', 'Coding', 'Quiz', 'Gaming', 'Music']
colleges_pool = [
    'St. Xavier Institute',
    'National Tech College',
    'Metro Arts Academy',
    'Apex Engineering College',
    'City Commerce College',
    'Global Science University'
]
ticket_tiers = list(range(0, 501, 50))

df_registrations = pd.DataFrame({
    'reg_id': reg_ids,
    'student_id': student_ids,
    'event': np.random.choice(events_pool, size=400),
    'college': np.random.choice(colleges_pool, size=400),
    'ticket_inr': np.random.choice(ticket_tiers, size=400),
    'paid': np.random.choice(['Yes', 'No'], size=400, p=[0.75, 0.25])
})

print(f"Generated Registrations DataFrame: {df_registrations.shape}")
print(df_registrations.head())


Generated Registrations DataFrame: (400, 6)
  reg_id student_id   event                    college  ticket_inr paid
0   R001       S001  Gaming         Metro Arts Academy         500  Yes
1   R002       S002   Music      City Commerce College         500   No
2   R003       S003    Quiz  Global Science University         500  Yes
3   R004       S004   Music       St. Xavier Institute         100  Yes
4   R005       S005   Music      City Commerce College         400  Yes


In [2]:
# Connect to SQLite database and write registrations table
conn = sqlite3.connect('fest.db')
df_registrations.to_sql('registrations', conn, if_exists='replace', index=False)

# Read paid rows using pd.read_sql_query
sql_query = "SELECT * FROM registrations WHERE paid = 'Yes'"
df_paid_query = pd.read_sql_query(sql_query, conn)

# Read paid rows using pd.read_sql
df_paid_sql = pd.read_sql(sql_query, conn)

print("=== SQL Query Ingestion Results ===")
print(f"pd.read_sql_query shape : {df_paid_query.shape} (Rows: {df_paid_query.shape[0]}, Cols: {df_paid_query.shape[1]})")
print(f"pd.read_sql shape       : {df_paid_sql.shape} (Rows: {df_paid_sql.shape[0]}, Cols: {df_paid_sql.shape[1]})")
print(f"Shapes Match Exactly    : {df_paid_query.shape == df_paid_sql.shape}")

# Close database connection
conn.close()


=== SQL Query Ingestion Results ===
pd.read_sql_query shape : (287, 6) (Rows: 287, Cols: 6)
pd.read_sql shape       : (287, 6) (Rows: 287, Cols: 6)
Shapes Match Exactly    : True


---
## Task 2: Nested API-Style JSON & `pd.json_normalize` Flattening

**Objectives:**
- Construct a realistic, nested API-style response payload for 15 fictional festival sponsors.
- Structure payload matching schema:
  `{'status': 'ok', 'data': [{'sponsor': ..., 'amount_inr': ..., 'contact': {'city': ..., 'tier': ...}}]}`
- Serialize to `sponsors_api.json` using `json.dump`.
- Ingest with `json.load` and unpack the semi-structured records into a clean 2D tabular DataFrame using `pd.json_normalize()`.
- Display columns and verify flattened fields: `contact.city` and `contact.tier`.


In [3]:
import json

# Define 15 corporate sponsors with nested contact metadata
sponsors_payload = {
    "status": "ok",
    "data": [
        {"sponsor": "RedBull Gaming", "amount_inr": 150000, "contact": {"city": "Mumbai", "tier": "Platinum"}},
        {"sponsor": "TechGig Hackathon", "amount_inr": 120000, "contact": {"city": "Bengaluru", "tier": "Platinum"}},
        {"sponsor": "Spotify India", "amount_inr": 100000, "contact": {"city": "Mumbai", "tier": "Gold"}},
        {"sponsor": "Domino's Pizza", "amount_inr": 80000, "contact": {"city": "Delhi", "tier": "Gold"}},
        {"sponsor": "Zepto Quick", "amount_inr": 75000, "contact": {"city": "Pune", "tier": "Gold"}},
        {"sponsor": "Boat Lifestyle", "amount_inr": 60000, "contact": {"city": "Delhi", "tier": "Silver"}},
        {"sponsor": "Unstop Campus", "amount_inr": 50000, "contact": {"city": "Bengaluru", "tier": "Silver"}},
        {"sponsor": "Nestle Nescafe", "amount_inr": 45000, "contact": {"city": "Kolkata", "tier": "Silver"}},
        {"sponsor": "Logitech G", "amount_inr": 40000, "contact": {"city": "Hyderabad", "tier": "Bronze"}},
        {"sponsor": "CodeChef Live", "amount_inr": 35000, "contact": {"city": "Chennai", "tier": "Bronze"}},
        {"sponsor": "Decathlon Sports", "amount_inr": 30000, "contact": {"city": "Ahmedabad", "tier": "Bronze"}},
        {"sponsor": "Keventers Shakes", "amount_inr": 25000, "contact": {"city": "Jaipur", "tier": "Bronze"}},
        {"sponsor": "Skillshare Learn", "amount_inr": 20000, "contact": {"city": "Noida", "tier": "Supporter"}},
        {"sponsor": "Subway Fresh", "amount_inr": 15000, "contact": {"city": "Gurugram", "tier": "Supporter"}},
        {"sponsor": "Duolingo Club", "amount_inr": 10000, "contact": {"city": "Chandigarh", "tier": "Supporter"}}
    ]
}

# Write nested API response to sponsors_api.json
with open('sponsors_api.json', 'w', encoding='utf-8') as f:
    json.dump(sponsors_payload, f, indent=2)

print("Saved nested API response to sponsors_api.json.")


Saved nested API response to sponsors_api.json.


In [4]:
# Load JSON and flatten semi-structured data using pd.json_normalize
with open('sponsors_api.json', 'r', encoding='utf-8') as f:
    api_response = json.load(f)

# Flatten 'data' key into 2D DataFrame
sponsors = pd.json_normalize(api_response['data'])

print("=== Flattened Sponsors DataFrame ===")
print("Columns:", sponsors.columns.tolist())
assert 'contact.city' in sponsors.columns, "contact.city must be present!"
assert 'contact.tier' in sponsors.columns, "contact.tier must be present!"
print("\nFirst 5 Records:")
print(sponsors.head())

print("\nSponsorship Breakdown by Tier:")
print(sponsors.groupby('contact.tier')['amount_inr'].agg(['count', 'sum']))


=== Flattened Sponsors DataFrame ===
Columns: ['sponsor', 'amount_inr', 'contact.city', 'contact.tier']

First 5 Records:
             sponsor  amount_inr contact.city contact.tier
0     RedBull Gaming      150000       Mumbai     Platinum
1  TechGig Hackathon      120000    Bengaluru     Platinum
2      Spotify India      100000       Mumbai         Gold
3     Domino's Pizza       80000        Delhi         Gold
4        Zepto Quick       75000         Pune         Gold

Sponsorship Breakdown by Tier:
              count     sum
contact.tier               
Bronze            4  130000
Gold              3  255000
Platinum          2  270000
Silver            3  155000
Supporter         3   45000


---
## Task 3: Event Catalog & Web-Based URL Ingestion

**Objectives:**
- Create `fest_events.csv` with 5 event rows:
  - Columns: `event`, `venue`, `day` (1 through 3), `budget_inr`.
  - Events: `Dance`, `Coding`, `Quiz`, `Gaming`, `Music`.
- Ingest `fest_events.csv` using its web URL on GitHub (`raw.githubusercontent.com`).
- Print `events.shape`.
- *Hint applied:* In GitHub, click **Raw** to get the direct raw content URL.


In [5]:
# 1. Create event catalog DataFrame
events_master_data = {
    'event': ['Dance', 'Coding', 'Quiz', 'Gaming', 'Music'],
    'venue': ['Main Auditorium', 'Computer Lab 3', 'Seminar Hall B', 'E-Sports Arena', 'Open Air Amphitheatre'],
    'day': [1, 2, 1, 3, 2],
    'budget_inr': [80000, 60000, 30000, 95000, 110000]
}
events_df = pd.DataFrame(events_master_data)

# Save to fest_events.csv
events_df.to_csv('fest_events.csv', index=False)
print("Saved fest_events.csv.")
print(events_df)


Saved fest_events.csv.
    event                  venue  day  budget_inr
0   Dance        Main Auditorium    1       80000
1  Coding         Computer Lab 3    2       60000
2    Quiz         Seminar Hall B    1       30000
3  Gaming         E-Sports Arena    3       95000
4   Music  Open Air Amphitheatre    2      110000


In [6]:
# 2. Ingest via raw.githubusercontent.com URL (with local fallback for offline resilience)
raw_url = "https://raw.githubusercontent.com/Atharvchaudhari1106/ds-work/main/assignment-3/s05_fest_collect/fest_events.csv"

try:
    events = pd.read_csv(raw_url)
    print(f"Successfully loaded from GitHub Raw URL: {raw_url}")
except Exception as e:
    print(f"Web fetch notice: Loading local fest_events.csv ({e})")
    events = pd.read_csv('fest_events.csv')

print(f"\nevents.shape: {events.shape} (5 events x 4 attributes)")
print(events)


Web fetch notice: Loading local fest_events.csv (HTTP Error 404: Not Found)

events.shape: (5, 4) (5 events x 4 attributes)
    event                  venue  day  budget_inr
0   Dance        Main Auditorium    1       80000
1  Coding         Computer Lab 3    2       60000
2    Quiz         Seminar Hall B    1       30000
3  Gaming         E-Sports Arena    3       95000
4   Music  Open Air Amphitheatre    2      110000


---
## Task 4: Pathlib Handling, Batch Globbing, Concat & Validation Merge

**Objectives:**
- Partition the 400 `registrations` into 3 daily batch files:
  - `fest_data/day1.csv`: 130 rows (`0:130`)
  - `fest_data/day2.csv`: 140 rows (`130:270`)
  - `fest_data/day3.csv`: 130 rows (`270:400`)
  - Total: $130 + 140 + 130 = 400$ rows.
  - Create directory using `Path('fest_data').mkdir(exist_ok=True)`.
- Discover and read all daily files dynamically using `sorted(Path('fest_data').glob('day*.csv'))`.
- Recombine using `pd.concat(..., ignore_index=True)`.
- Execute a validated relational merge with `events` on `event`:
  - `pd.merge(recombined, events, on='event', how='left', validate='m:1')`
- Export master dataset to `fest_master.csv` (`index=False`).
- Assert:
  1. `len(fest_master) == 400`
  2. `fest_master['venue'].isna().sum() == 0` (no unmapped event venues).


In [7]:
from pathlib import Path

# 1. Create directory using Pathlib
data_dir = Path('fest_data')
data_dir.mkdir(exist_ok=True)

# 2. Partition registrations into 3 daily splits (130, 140, 130 rows)
day1_df = df_registrations.iloc[:130]
day2_df = df_registrations.iloc[130:270]
day3_df = df_registrations.iloc[270:400]

day1_df.to_csv(data_dir / 'day1.csv', index=False)
day2_df.to_csv(data_dir / 'day2.csv', index=False)
day3_df.to_csv(data_dir / 'day3.csv', index=False)

print(f"Partitioned registrations into {data_dir}/:")
print(f" - day1.csv : {len(day1_df)} rows")
print(f" - day2.csv : {len(day2_df)} rows")
print(f" - day3.csv : {len(day3_df)} rows")


Partitioned registrations into fest_data/:
 - day1.csv : 130 rows
 - day2.csv : 140 rows
 - day3.csv : 130 rows


In [8]:
# 3. Dynamic batch discovery & horizontal concatenation
csv_files = sorted(Path('fest_data').glob('day*.csv'))
print(f"Discovered {len(csv_files)} files: {[f.name for f in csv_files]}")

# Load and concatenate
daily_dfs = [pd.read_csv(f) for f in csv_files]
recombined_registrations = pd.concat(daily_dfs, ignore_index=True)

print(f"Recombined DataFrame Shape: {recombined_registrations.shape}")


Discovered 3 files: ['day1.csv', 'day2.csv', 'day3.csv']
Recombined DataFrame Shape: (400, 6)


In [9]:
# 4. Validated Left Merge: Many-to-One (registrations:events)
fest_master = pd.merge(
    recombined_registrations,
    events,
    on='event',
    how='left',
    validate='m:1'
)

# Export master festival dataset
fest_master.to_csv('fest_master.csv', index=False)

print("=== fest_master.csv Summary ===")
print(f"Master Shape : {fest_master.shape} (400 rows x 9 columns)")
print("\nFirst 5 Records of fest_master:")
print(fest_master.head())

# Rigorous Quality Assertions
assert len(fest_master) == 400, f"Expected 400 rows, found {len(fest_master)}!"
missing_venues = fest_master['venue'].isna().sum()
assert missing_venues == 0, f"Found {missing_venues} unmapped venues!"

print(f"\nAssertions Verified:")
print(f"  [PASS] Total rows equal exactly 400: {len(fest_master) == 400}")
print(f"  [PASS] Zero missing venues in master dataset: {missing_venues == 0}")


=== fest_master.csv Summary ===
Master Shape : (400, 9) (400 rows x 9 columns)

First 5 Records of fest_master:
  reg_id student_id   event  ...                  venue  day  budget_inr
0   R001       S001  Gaming  ...         E-Sports Arena    3       95000
1   R002       S002   Music  ...  Open Air Amphitheatre    2      110000
2   R003       S003    Quiz  ...         Seminar Hall B    1       30000
3   R004       S004   Music  ...  Open Air Amphitheatre    2      110000
4   R005       S005   Music  ...  Open Air Amphitheatre    2      110000

[5 rows x 9 columns]

Assertions Verified:
  [PASS] Total rows equal exactly 400: True
  [PASS] Zero missing venues in master dataset: True


---
## Summary of Key Takeaways:

1. **Database Connectors (`read_sql` vs `read_sql_query`):**
   - `read_sql_query()` executes explicit SQL queries with parameter safety.
   - `read_sql()` is a convenience dispatcher that accepts both queries and table names.
2. **Semi-Structured Flattening with `pd.json_normalize`:**
   - Real-world REST APIs frequently return hierarchical nested payloads.
   - `json_normalize` traverses nested dictionaries and flattens keys using dot-notation (`contact.city`), preventing manual unpacking loops.
3. **Web-Based Data Ingestion:**
   - Pandas parses remote URLs seamlessly over HTTP/HTTPS directly into memory buffers, eliminating manual file downloading.
4. **`pathlib` over `os.path`:**
   - Modern Python uses object-oriented `Path` objects (`/` operator, `.glob()`, `.mkdir()`) which are platform-independent and cleaner than legacy string concatenation.
5. **Relational Merge Validation (`validate='m:1'`):**
   - Specifying merge cardinality constraints (`m:1`, `1:m`, `1:1`) catches subtle Cartesian explosion bugs early before corrupted joins propagate downstream.
